# Build the London MSOA modelling dataset

This notebook combines the final MSOA-level datasets used in the analysis:
- 2021 MSOA names, boroughs and areas
- mid-2024 population estimates
- active NaPTAN transport-node counts
- final classified bakery establishment counts

In [1]:
from pathlib import Path

import geopandas as gpd
import pandas as pd


DATA_DATE = "2026-07-23"

SPATIAL_PROCESSED_FOLDER = Path("../data/spatial/processed")

MSOA_PATH = (SPATIAL_PROCESSED_FOLDER / "london_msoa_2021.gpkg")
POPULATION_PATH = (SPATIAL_PROCESSED_FOLDER / "london_msoa_population_2024.csv")
TRANSPORT_PATH = (SPATIAL_PROCESSED_FOLDER / "london_msoa_transport_nodes.csv")
BAKERY_COUNTS_PATH = (SPATIAL_PROCESSED_FOLDER / f"london_msoa_bakery_counts_{DATA_DATE}.csv")
BAKERY_ESTABLISHMENTS_PATH = (SPATIAL_PROCESSED_FOLDER / f"london_bakery_establishments_with_msoa_{DATA_DATE}.csv")
OUTPUT_PATH = (SPATIAL_PROCESSED_FOLDER / f"london_msoa_modelling_dataset_{DATA_DATE}.csv")

EXPECTED_MSOAS = 1002
EXPECTED_MATCHED_BAKERIES = 5178

## Load the processed MSOA datasets

Only the fields required for the final modelling dataset are loaded.

In [2]:
msoa = gpd.read_file(MSOA_PATH, ignore_geometry=True)

msoa = msoa[["msoa_code",
             "msoa_name",
             "borough_code",
             "borough_name",
             "area_km2"]].copy()

population = pd.read_csv(POPULATION_PATH, usecols=["msoa_code", "population"])
transport = pd.read_csv(TRANSPORT_PATH, usecols=["msoa_code", "transport_nodes"])
bakery_counts = pd.read_csv(BAKERY_COUNTS_PATH, usecols=["msoa_code", "bakery_count"])
bakery_establishments = pd.read_csv(BAKERY_ESTABLISHMENTS_PATH,
                                    usecols=["FHRSID", "msoa_code", "FinalClass"],
                                    low_memory=False)

print(f"MSOA rows: {len(msoa)}")
print(f"Population rows: {len(population)}")
print(f"Transport rows: {len(transport)}")
print(f"Bakery-count rows: {len(bakery_counts)}")

print(f"\nBakery establishment rows: {len(bakery_establishments)}")
print(f"Bakery establishments with an MSOA: {bakery_establishments["msoa_code"].notna().sum()}")

MSOA rows: 1002
Population rows: 1002
Transport rows: 1002
Bakery-count rows: 1002

Bakery establishment rows: 5221
Bakery establishments with an MSOA: 5178


In [3]:
matched_bakeries = (bakery_establishments
                    .dropna(subset=["msoa_code"])
                    .copy())

matched_bakeries["core_bakery_count"] = (matched_bakeries["FinalClass"]
                                         .eq("CORE_BAKERY")
                                         .astype(int))

matched_bakeries["core_and_cafe_count"] = (matched_bakeries["FinalClass"]
                                           .isin(["CORE_BAKERY", "BAKERY_CAFE"])
                                           .astype(int))

bakery_definition_counts = (matched_bakeries
                            .groupby("msoa_code")[["core_bakery_count",
                                                   "core_and_cafe_count"]]
                                                   .sum()
                                                   .reset_index())

print(f"Spatially assigned core bakeries: {bakery_definition_counts["core_bakery_count"].sum()}")
print(f"Spatially assigned core and café bakeries: {bakery_definition_counts["core_and_cafe_count"].sum()}")
print(f"Spatially assigned all bakery establishments: {len(matched_bakeries)}")

Spatially assigned core bakeries: 925
Spatially assigned core and café bakeries: 2928
Spatially assigned all bakery establishments: 5178


## Combine the MSOA-level datasets

The MSOA boundary table is used so that all 1,002 Greater London MSOAs are retained. Population, transport and bakery counts are joined using MSOA code.

In [4]:
msoa_modelling = (msoa
                  .merge(population,
                         on="msoa_code",
                         how="left",
                         validate="one_to_one")
                         .merge(transport,
                                on="msoa_code",
                                how="left",
                                validate="one_to_one")
                                .merge(bakery_counts,
                                       on="msoa_code",
                                       how="left",
                                       validate="one_to_one")
                                       .merge(bakery_definition_counts,
                                              on="msoa_code",
                                              how="left",
                                              validate="one_to_one")
                                              .sort_values("msoa_code")
                                              .reset_index(drop=True))

definition_columns = ["core_bakery_count", "core_and_cafe_count"]

msoa_modelling[definition_columns] = (msoa_modelling[definition_columns]
                                      .fillna(0)
                                      .astype(int))

print(f"Combined MSOA rows: {len(msoa_modelling)}")

display(msoa_modelling.head())

Combined MSOA rows: 1002


,msoa_code,msoa_name,borough_code,borough_name,area_km2,population,transport_nodes,bakery_count,core_bakery_count,core_and_cafe_count
0,E02000001,City of London 001,E09000001,City of London,3.150420,15111,235,129,13,114
1,E02000002,Barking and Dagenham 001,E09000002,Barking and Dagenham,2.161561,8820,20,0,0,0
2,E02000003,Barking and Dagenham 002,E09000002,Barking and Dagenham,2.141515,12655,21,7,1,1
3,E02000004,Barking and Dagenham 003,E09000002,Barking and Dagenham,2.492946,7056,13,4,0,1
4,E02000005,Barking and Dagenham 004,E09000002,Barking and Dagenham,1.187954,11630,11,4,0,1


## Calculate provision and density measures

Several measures are calculated for the descriptive and spatial analysis:

- population per square kilometre
- transport nodes per square kilometre
- bakeries per square kilometre
- bakeries per 10,000 residents

Original counts are also retained as bakery count is the outcome used in the statistical model.

In [5]:
msoa_modelling["population_density_km2"] = (msoa_modelling["population"] / msoa_modelling["area_km2"])
msoa_modelling["transport_nodes_per_km2"] = (msoa_modelling["transport_nodes"] / msoa_modelling["area_km2"])
msoa_modelling["bakery_density_km2"] = (msoa_modelling["bakery_count"] / msoa_modelling["area_km2"])
msoa_modelling["bakeries_per_10000_people"] = (msoa_modelling["bakery_count"] / msoa_modelling["population"] * 10000)
msoa_modelling["core_bakeries_per_10000_people"] = (msoa_modelling["core_bakery_count"] / msoa_modelling["population"]* 10000)
msoa_modelling["core_and_cafe_per_10000_people"] = (msoa_modelling["core_and_cafe_count"] / msoa_modelling["population"] * 10000)

display(
    msoa_modelling[["area_km2",
                    "population",
                    "transport_nodes",
                    "bakery_count",
                    "population_density_km2",
                    "transport_nodes_per_km2",
                    "bakery_density_km2",
                    "bakeries_per_10000_people"]]
                    .describe()
                    .round(2))

,area_km2,population,transport_nodes,bakery_count,population_density_km2,transport_nodes_per_km2,bakery_density_km2,bakeries_per_10000_people
count,1002.00,1002.00,1002.00,1002.00,1002.00,1002.00,1002.00,1002.00
mean,1.59,9071.59,22.78,5.17,9020.36,19.29,5.49,5.76
std,1.87,1958.13,15.27,7.90,5042.87,11.37,7.81,8.81
min,0.29,5274.00,0.00,0.00,302.81,0.00,0.00,0.00
25%,0.71,7684.00,14.00,1.00,5130.55,11.74,0.90,1.37
50%,1.13,8747.00,20.00,3.00,8142.37,16.72,2.58,3.63
75%,1.73,10122.75,28.00,7.00,12432.00,23.96,6.63,7.06
max,22.43,16767.00,235.00,129.00,27449.84,96.39,71.18,142.31


In [6]:
definition_summary = pd.DataFrame({"Definition": ["Core only",
                                                  "Core and café",
                                                  "All including supermarkets"],
                                   "Establishments": [msoa_modelling["core_bakery_count"].sum(),
                                                      msoa_modelling["core_and_cafe_count"].sum(),
                                                      msoa_modelling["bakery_count"].sum()],
                                   "Mean per MSOA": [msoa_modelling["core_bakery_count"].mean(),
                                                      msoa_modelling["core_and_cafe_count"].mean(),
                                                      msoa_modelling["bakery_count"].mean()],
                                   "MSOAs with zero": [msoa_modelling["core_bakery_count"].eq(0).sum(),
                                                       msoa_modelling["core_and_cafe_count"].eq(0).sum(),
                                                       msoa_modelling["bakery_count"].eq(0).sum()],
                                   "Mean per 10,000 people": [msoa_modelling["core_bakeries_per_10000_people"].mean(),
                                                              msoa_modelling["core_and_cafe_per_10000_people"].mean(),
                                                              msoa_modelling["bakeries_per_10000_people"].mean()]})

definition_summary[["Mean per MSOA", "Mean per 10,000 people"]
                   ] = (definition_summary[["Mean per MSOA", "Mean per 10,000 people"]
                                           ].round(2))

display(definition_summary)

,Definition,Establishments,Mean per MSOA,MSOAs with zero,"Mean per 10,000 people"
0,Core only,925,0.92,557,1.05
1,Core and café,2928,2.92,304,3.31
2,All including supermarkets,5178,5.17,120,5.76


## Validate the modelling dataset

A final set of checks to confirm all London MSOAs are represented once, required fields are available and bakery counts agree with the spatial assignment in earlier notebooks.

In [9]:
required_columns = ["msoa_code",
                    "msoa_name",
                    "borough_code",
                    "borough_name",
                    "area_km2",
                    "population",
                    "transport_nodes",
                    "bakery_count",
                    "population_density_km2",
                    "transport_nodes_per_km2",
                    "bakery_density_km2",
                    "bakeries_per_10000_people",
                    "core_bakery_count",
                    "core_and_cafe_count",
                    "core_bakeries_per_10000_people",
                    "core_and_cafe_per_10000_people"]

missing_values = (msoa_modelling[required_columns].isna().sum())

print(f"Final MSOA rows: {len(msoa_modelling)}")
print(f"Unique MSOA codes: {msoa_modelling["msoa_code"].nunique()}")
print(f"Duplicate MSOA codes: {msoa_modelling["msoa_code"].duplicated().sum()}")

print("\nMissing values:")
display(missing_values.rename("Missing"))

print(f"Total London population: {msoa_modelling["population"].sum()}")
print(f"Total transport nodes: {msoa_modelling["transport_nodes"].sum()}")
print(f"Core bakeries: {msoa_modelling["core_bakery_count"].sum()}")
print(f"Core and café bakeries: {msoa_modelling["core_and_cafe_count"].sum()}")
print(f"All bakeries including supermarkets: {msoa_modelling["bakery_count"].sum()}")

print(f"\nMSOAs with no core bakery: {msoa_modelling["core_bakery_count"].eq(0).sum()}")
print(f"MSOAs with no core or café bakery: {msoa_modelling["core_and_cafe_count"].eq(0).sum()}")
print(f"MSOAs with no included bakery: {msoa_modelling["bakery_count"].eq(0).sum()}")

if len(msoa_modelling) != EXPECTED_MSOAS:
    raise RuntimeError(f"Expected {EXPECTED_MSOAS} MSOA rows, but found {len(msoa_modelling)}.")

if msoa_modelling["msoa_code"].duplicated().any():
    raise RuntimeError("Duplicate MSOA codes found in the modelling dataset.")

if missing_values.sum() > 0:
    raise RuntimeError("Missing values found in the modelling dataset.")

if (msoa_modelling["bakery_count"].sum() != EXPECTED_MATCHED_BAKERIES):
    raise RuntimeError("Bakery counts do not agree with notebook 12.")

invalid_definition_order = (( msoa_modelling["core_bakery_count"] > msoa_modelling["core_and_cafe_count"])
                            |(msoa_modelling["core_and_cafe_count"] > msoa_modelling["bakery_count"]))

print(f"\nInvalid cumulative bakery counts: {invalid_definition_order.sum()}")

if invalid_definition_order.any():
    raise RuntimeError("The cumulative bakery definitions are inconsistent.")

print("\nFinal modelling dataset validation passed.")

Final MSOA rows: 1002
Unique MSOA codes: 1002
Duplicate MSOA codes: 0

Missing values:


msoa_code                         0
msoa_name                         0
borough_code                      0
borough_name                      0
area_km2                          0
population                        0
transport_nodes                   0
bakery_count                      0
population_density_km2            0
transport_nodes_per_km2           0
bakery_density_km2                0
bakeries_per_10000_people         0
core_bakery_count                 0
core_and_cafe_count               0
core_bakeries_per_10000_people    0
core_and_cafe_per_10000_people    0
Name: Missing, dtype: int64

Total London population: 9089736
Total transport nodes: 22821
Core bakeries: 925
Core and café bakeries: 2928
All bakeries including supermarkets: 5178

MSOAs with no core bakery: 557
MSOAs with no core or café bakery: 304
MSOAs with no included bakery: 120

Invalid cumulative bakery counts: 0

Final modelling dataset validation passed.


## Saving final modelling dataset

In [8]:
msoa_modelling.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")

print(f"Saved modelling dataset to: {OUTPUT_PATH}")
print(f"Final rows: {len(msoa_modelling)}")
print(f"Final columns: {len(msoa_modelling.columns)}")

Saved modelling dataset to: ..\data\spatial\processed\london_msoa_modelling_dataset_2026-07-23.csv
Final rows: 1002
Final columns: 16
